# Modulo 3 · Limpieza y transformacion de datos

En el mundo real los datos casi nunca llegan perfectos. En este modulo vas a
detectar y corregir problemas comunes: valores nulos, filas duplicadas,
tipos de datos incorrectos, y vas a crear columnas nuevas a partir de las
existentes (feature engineering).

**Contenidos:**
1. Detectar valores nulos
2. Tratar valores nulos
3. Detectar y eliminar duplicados
4. Corregir tipos de datos
5. Crear columnas nuevas (feature engineering de fechas)


In [1]:
import pandas as pd

df = pd.read_csv("../data/ventas.csv", parse_dates=["fecha"])
df.shape


(12030, 13)

## 1. Detectar valores nulos

In [2]:
# Cuantos valores nulos hay por columna
df.isna().sum()


id_venta              0
fecha                 0
region                0
categoria             0
producto              0
vendedor              0
segmento_cliente    241
unidades              0
precio_unitario       0
descuento_pct         0
ingreso               0
costo                 0
utilidad              0
dtype: int64

In [3]:
# Porcentaje de nulos por columna
(df.isna().mean() * 100).round(2)


id_venta            0.0
fecha               0.0
region              0.0
categoria           0.0
producto            0.0
vendedor            0.0
segmento_cliente    2.0
unidades            0.0
precio_unitario     0.0
descuento_pct       0.0
ingreso             0.0
costo               0.0
utilidad            0.0
dtype: float64

La columna `segmento_cliente` tiene valores nulos: registros donde no se
capturo el segmento del cliente. Veamos algunas de esas filas.

In [4]:
df[df["segmento_cliente"].isna()].head()


,id_venta,fecha,region,categoria,producto,vendedor,segmento_cliente,unidades,precio_unitario,descuento_pct,ingreso,costo,utilidad
112,113,2023-01-07,Norte,Electronica,Cargador,Elena Vidal,NaN,4,8960.0,0.20,28672.0,21504.0,7168.0
113,114,2023-01-07,Oeste,Deportes,Colchoneta Yoga,Ana Torres,NaN,4,9510.0,0.00,38040.0,22824.0,15216.0
181,182,2023-01-12,Centro,Ropa,Chaqueta,Isidora Munoz,NaN,1,32570.0,0.15,27684.0,19542.0,8142.0
210,211,2023-01-14,Sur,Ropa,Gorra,Ana Torres,NaN,7,5980.0,0.20,33488.0,25116.0,8372.0
385,386,2023-01-25,Sur,Electronica,Teclado,Ana Torres,NaN,2,12940.0,0.05,24586.0,15528.0,9058.0


## 2. Tratar valores nulos

Hay varias estrategias para tratar nulos:
- **Eliminar** las filas o columnas (si son pocas y no aportan valor).
- **Rellenar** con un valor por defecto (ej: `"Desconocido"`).
- **Imputar** con la moda, media o mediana (para numericos).

Para `segmento_cliente`, como es una variable categorica y no queremos perder
las ventas (que si tienen ingreso valido), la rellenamos con `"Desconocido"`.

In [5]:
df["segmento_cliente"] = df["segmento_cliente"].fillna("Desconocido")
df["segmento_cliente"].value_counts()


segmento_cliente
Individual     6454
Pyme           3527
Corporativo    1808
Desconocido     241
Name: count, dtype: int64

## 3. Detectar y eliminar duplicados

In [6]:
# Contar filas duplicadas exactas
print(f"Filas duplicadas: {df.duplicated().sum()}")
df[df.duplicated(keep=False)].sort_values("id_venta").head(10)


Filas duplicadas: 0


,id_venta,fecha,region,categoria,producto,vendedor,segmento_cliente,unidades,precio_unitario,descuento_pct,ingreso,costo,utilidad


In [7]:
filas_antes = len(df)
df = df.drop_duplicates()
filas_despues = len(df)
print(f"Se eliminaron {filas_antes - filas_despues} filas duplicadas")
print(f"Filas restantes: {filas_despues}")


Se eliminaron 0 filas duplicadas
Filas restantes: 12030


## 4. Corregir tipos de datos

In [8]:
df.dtypes


id_venta                     int64
fecha               datetime64[us]
region                         str
categoria                      str
producto                       str
vendedor                       str
segmento_cliente               str
unidades                     int64
precio_unitario            float64
descuento_pct              float64
ingreso                    float64
costo                      float64
utilidad                   float64
dtype: object

In [9]:
# 'fecha' ya quedo como datetime porque usamos parse_dates al leer el CSV.
# Si no lo hubieramos hecho, se convierte asi:
# df["fecha"] = pd.to_datetime(df["fecha"])

# region, categoria, producto, vendedor y segmento_cliente son categoricas de baja cardinalidad:
# convertirlas a tipo 'category' ahorra memoria y acelera agrupaciones.
columnas_categoricas = ["region", "categoria", "producto", "vendedor", "segmento_cliente"]
for col in columnas_categoricas:
    df[col] = df[col].astype("category")

df.dtypes


id_venta                     int64
fecha               datetime64[us]
region                    category
categoria                 category
producto                  category
vendedor                  category
segmento_cliente          category
unidades                     int64
precio_unitario            float64
descuento_pct              float64
ingreso                    float64
costo                      float64
utilidad                   float64
dtype: object

## 5. Crear columnas nuevas (feature engineering)

In [10]:
# Extraer partes de la fecha para poder analizar tendencias
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["nombre_mes"] = df["fecha"].dt.month_name()
df["dia_semana"] = df["fecha"].dt.day_name()
df["trimestre"] = df["fecha"].dt.quarter

df[["fecha", "anio", "mes", "nombre_mes", "dia_semana", "trimestre"]].head()


,fecha,anio,mes,nombre_mes,dia_semana,trimestre
0,2023-01-01,2023,1,January,Sunday,1
1,2023-01-01,2023,1,January,Sunday,1
2,2023-01-01,2023,1,January,Sunday,1
3,2023-01-01,2023,1,January,Sunday,1
4,2023-01-01,2023,1,January,Sunday,1


In [11]:
# Margen de utilidad porcentual por venta
df["margen_pct"] = (df["utilidad"] / df["ingreso"]).round(4)
df[["producto", "ingreso", "costo", "utilidad", "margen_pct"]].head()


,producto,ingreso,costo,utilidad,margen_pct
0,Bicicleta,664760.0,498570.0,166190.0,0.2500
1,Parlante Bluetooth,50840.0,30504.0,20336.0,0.4000
2,Aspiradora,409581.0,289116.0,120465.0,0.2941
3,Aspiradora,388150.0,232890.0,155260.0,0.4000
4,Escritorio,585720.0,351432.0,234288.0,0.4000


In [12]:
# Verificacion final: ya no deberia haber nulos ni duplicados
print("Nulos totales:", df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("Forma final:", df.shape)


Nulos totales: 0
Duplicados: 0
Forma final: (12030, 19)


Guardamos el dataset limpio para usarlo en los siguientes modulos, sin tener
que repetir estos pasos cada vez.

In [13]:
df.to_csv("../data/ventas_limpio.csv", index=False)
print("Guardado en data/ventas_limpio.csv")


Guardado en data/ventas_limpio.csv


## 🧠 Retos del modulo 3

1. ¿Cual es el rango de fechas del dataset (fecha minima y maxima)?
2. Crea una columna `es_fin_de_semana` (booleana) que sea `True` si la venta
   ocurrio sabado o domingo.
3. ¿Cuantas ventas tienen `margen_pct` negativo o igual a cero? Investiga por que
   podria pasar esto (pista: revisa `descuento_pct`).
4. Crea una columna `rango_ingreso` que clasifique cada venta en `"Bajo"` (< 30.000),
   `"Medio"` (30.000 - 100.000) o `"Alto"` (> 100.000) usando `pd.cut` o una funcion propia.

Compara tus resultados con `solutions/03_limpieza_datos_solucion.py`.


In [14]:
# Escribe aqui tu solucion


